# PyTorch GRU baseline (CPU)

Minimal sequence model for light curves with masking.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import precision_recall_curve
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = Path('datasets/')
FILTERS = ['u', 'g', 'r', 'i', 'z', 'y']
FILTER_TO_ID = {f: i for i, f in enumerate(FILTERS)}
DEVICE = 'cpu'

torch.set_num_threads(4)


In [ ]:
@dataclass
class Config:
    seed: int = 42
    n_folds: int = 5
    batch_size: int = 64
    epochs: int = 8
    lr: float = 1e-3
    hidden: int = 64

def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)

cfg = Config()
set_seed(cfg.seed)


In [ ]:
def load_meta():
    train_log = pd.read_csv(DATA_ROOT / 'train_log.csv')
    test_log = pd.read_csv(DATA_ROOT / 'test_log.csv')
    return train_log, test_log

def get_splits(train_log: pd.DataFrame, n_folds: int = 5):
    splits = sorted(train_log['split'].unique())
    folds = []
    for i in range(n_folds):
        folds.append(splits[i::n_folds])
    return folds

def load_lightcurves_for_splits(splits: list[str], kind: str) -> pd.DataFrame:
    parts = []
    for split in splits:
        fname = 'train_full_lightcurves.csv' if kind == 'train' else 'test_full_lightcurves.csv'
        path = DATA_ROOT / split / fname
        df = pd.read_csv(path)
        df['split'] = split
        parts.append(df)
    return pd.concat(parts, ignore_index=True)

def clean_lightcurves(lc: pd.DataFrame) -> pd.DataFrame:
    lc = lc.copy()
    lc = lc[lc['Flux'].notna()]
    lc['Time (MJD)'] = lc['Time (MJD)'].astype(float)
    lc['Flux'] = lc['Flux'].astype(float)
    lc['Flux_err'] = lc['Flux_err'].astype(float)
    return lc

def add_time_features(lc: pd.DataFrame) -> pd.DataFrame:
    lc = lc.copy()
    lc['t0'] = lc.groupby('object_id')['Time (MJD)'].transform('min')
    lc['dt'] = lc['Time (MJD)'] - lc['t0']
    return lc


In [ ]:
class LightCurveSeqDataset(Dataset):
    def __init__(self, lc_df: pd.DataFrame, meta_df: pd.DataFrame):
        self.meta = meta_df.set_index('object_id')
        self.groups = lc_df.groupby('object_id')
        self.object_ids = list(self.groups.groups.keys())

    def __len__(self) -> int:
        return len(self.object_ids)

    def __getitem__(self, idx: int):
        oid = self.object_ids[idx]
        grp = self.groups.get_group(oid).copy()
        grp = grp.sort_values('dt')
        dt = grp['dt'].to_numpy(dtype=np.float32)
        flux = grp['Flux'].to_numpy(dtype=np.float32)
        ferr = grp['Flux_err'].to_numpy(dtype=np.float32)
        fid = grp['Filter'].map(FILTER_TO_ID).to_numpy(dtype=np.int64)

        # per-object robust normalization
        med = np.nanmedian(flux)
        mad = np.nanmedian(np.abs(flux - med))
        scale = mad if mad > 0 else np.nanstd(flux)
        if not np.isfinite(scale) or scale == 0:
            flux_z = flux
        else:
            flux_z = (flux - med) / scale

        x = np.stack([dt, flux_z, ferr, fid], axis=1)

        # meta features
        meta = np.zeros(2, dtype=np.float32)
        if oid in self.meta.index:
            meta[0] = float(self.meta.loc[oid, 'Z'])
            meta[1] = float(self.meta.loc[oid, 'EBV'])

        y = None
        if 'target' in self.meta.columns:
            y = int(self.meta.loc[oid, 'target'])

        return oid, x, meta, y

def collate_fn(batch):
    oids, xs, metas, ys = zip(*batch)
    lengths = [len(x) for x in xs]
    max_len = max(lengths)
    x_pad = np.zeros((len(xs), max_len, xs[0].shape[1]), dtype=np.float32)
    mask = np.zeros((len(xs), max_len), dtype=np.float32)
    for i, x in enumerate(xs):
        l = len(x)
        x_pad[i, :l] = x
        mask[i, :l] = 1.0
    x_pad = torch.tensor(x_pad)
    mask = torch.tensor(mask)
    meta = torch.tensor(np.stack(metas, axis=0), dtype=torch.float32)
    y = None if ys[0] is None else torch.tensor(ys, dtype=torch.float32)
    return oids, x_pad, mask, meta, y


In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, num_filters: int = 6, hidden: int = 128):
        super().__init__()
        self.embed = nn.Embedding(num_filters, 6)
        self.gru = nn.GRU(input_size=3 + 6, hidden_size=hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.meta_head = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden + 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, x, mask, meta):
        dt = x[..., 0:1]
        flux = x[..., 1:2]
        ferr = x[..., 2:3]
        fid = x[..., 3].long()
        fid_emb = self.embed(fid)
        feats = torch.cat([dt, flux, ferr, fid_emb], dim=-1)
        lengths = mask.sum(dim=1).long().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            feats, lengths, batch_first=True, enforce_sorted=False
        )
        _, h = self.gru(packed)
        h = h[-1]
        meta_feat = self.meta_head(meta)
        out = torch.cat([h, meta_feat], dim=1)
        logits = self.head(out).squeeze(-1)
        return logits


In [ ]:
def best_f1_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    idx = np.nanargmax(f1)
    return thresholds[idx], f1[idx], precision[idx], recall[idx]

def train_one_epoch(model, loader, optimizer, pos_weight):
    model.train()
    total_loss = 0.0
    for _, x, mask, meta, y in loader:
        x, mask, meta, y = x.to(DEVICE), mask.to(DEVICE), meta.to(DEVICE), y.to(DEVICE)
        logits = model(x, mask, meta)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    all_y = []
    all_p = []
    for _, x, mask, meta, y in loader:
        x, mask, meta = x.to(DEVICE), mask.to(DEVICE), meta.to(DEVICE)
        logits = model(x, mask, meta)
        prob = torch.sigmoid(logits).cpu().numpy()
        all_p.append(prob)
        all_y.append(y.numpy())
    all_p = np.concatenate(all_p)
    all_y = np.concatenate(all_y)
    thr, f1, p, r = best_f1_threshold(all_y, all_p)
    return f1, thr, p, r

@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    all_oids = []
    all_p = []
    for oids, x, mask, meta, _ in loader:
        x, mask, meta = x.to(DEVICE), mask.to(DEVICE), meta.to(DEVICE)
        logits = model(x, mask, meta)
        prob = torch.sigmoid(logits).cpu().numpy()
        all_oids.extend(oids)
        all_p.append(prob)
    all_p = np.concatenate(all_p)
    return np.array(all_oids), all_p


In [ ]:
train_log, test_log = load_meta()
folds = get_splits(train_log, n_folds=cfg.n_folds)
all_splits = sorted(train_log['split'].unique())

oof_pred = np.full(len(train_log), np.nan)
oof_true = train_log['target'].astype(int).to_numpy()
id_to_idx = {oid: i for i, oid in enumerate(train_log['object_id'].values)}

for fold_id, val_splits in enumerate(folds):
    print(f'FOLD {fold_id} val_splits: {val_splits}')
    train_splits = [s for s in all_splits if s not in val_splits]
    train_meta = train_log[train_log['split'].isin(train_splits)].copy()
    val_meta = train_log[train_log['split'].isin(val_splits)].copy()

    train_lc = load_lightcurves_for_splits(train_splits, kind='train')
    val_lc = load_lightcurves_for_splits(val_splits, kind='train')
    train_lc = add_time_features(clean_lightcurves(train_lc))
    val_lc = add_time_features(clean_lightcurves(val_lc))

    train_ds = LightCurveSeqDataset(train_lc, train_meta)
    val_ds = LightCurveSeqDataset(val_lc, val_meta)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

    model = GRUClassifier(num_filters=len(FILTERS), hidden=cfg.hidden).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    pos = train_meta['target'].sum()
    neg = len(train_meta) - pos
    pos_weight = torch.tensor([neg / max(pos, 1)], device=DEVICE)

    for epoch in range(cfg.epochs):
        loss = train_one_epoch(model, train_loader, optimizer, pos_weight)
        f1, thr, p, r = eval_model(model, val_loader)
        print(f'  epoch {epoch+1}: loss={loss:.4f} f1={f1:.4f} thr={thr:.3f} p={p:.3f} r={r:.3f}')

    oids, probs = predict_model(model, val_loader)
    for oid, prob in zip(oids, probs):
        oof_pred[id_to_idx[oid]] = prob

mask = np.isfinite(oof_pred)
oof_thr, oof_f1, oof_p, oof_r = best_f1_threshold(oof_true[mask], oof_pred[mask])
print(f'OOF Best F1={oof_f1:.4f}, thr={oof_thr:.3f}, P={oof_p:.4f}, R={oof_r:.4f}')


In [ ]:
# Full training + submission
train_lc = load_lightcurves_for_splits(all_splits, kind='train')
train_lc = add_time_features(clean_lightcurves(train_lc))
test_lc = load_lightcurves_for_splits(sorted(test_log['split'].unique()), kind='test')
test_lc = add_time_features(clean_lightcurves(test_lc))

full_train_ds = LightCurveSeqDataset(train_lc, train_log)
test_ds = LightCurveSeqDataset(test_lc, test_log)
full_train_loader = DataLoader(full_train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

model = GRUClassifier(num_filters=len(FILTERS), hidden=cfg.hidden).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

pos = train_log['target'].sum()
neg = len(train_log) - pos
pos_weight = torch.tensor([neg / max(pos, 1)], device=DEVICE)

for epoch in range(cfg.epochs):
    loss = train_one_epoch(model, full_train_loader, optimizer, pos_weight)
    print(f'full epoch {epoch+1}: loss={loss:.4f}')

oids, probs = predict_model(model, test_loader)
pred = (probs >= oof_thr).astype(int)
submission = pd.DataFrame({'object_id': oids, 'prediction': pred})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv, positives:', int(pred.sum()))
